In [1]:
import warnings
warnings.filterwarnings("ignore")


import pandas as pd

# Load data
df = pd.read_excel("/kaggle/input/datasets/desaluhorsamideksa/agri-dataset-1/ethiopia_agriculture_dataset.xlsx")

# Feature engineering
df["revenue_per_hectare"] = df["total_revenue_birr"] / df["land_hectares"]

df["rainfall_category"] = pd.cut(
    df["rainfall_mm"],
    bins=[0,700,1200,2000],
    labels=["Low","Medium","High"]
)

# -----------------------------
# ANALYSIS
# -----------------------------

# Rainfall impact
group = df.groupby("rainfall_category", observed=False)["revenue_per_hectare"].mean()

low = group["Low"]     # ✅ define low FIRST
high = group["High"]   # then high

rainfall_percent = ((high - low) / low) * 100



# Fertilizer impact
fert = df.groupby("fertilizer_use")["yield_per_hectare"].mean()
fert_percent = ((fert[1] - fert[0]) / fert[0]) * 100

# Percentile
df["percentile"] = df["revenue_per_hectare"].rank(pct=True)
top_10 = df[df["percentile"] > 0.9]

top10_share = (top_10["total_revenue_birr"].sum() /
               df["total_revenue_birr"].sum()) * 100

# Outliers
outliers = df[df["revenue_per_hectare"] > df["revenue_per_hectare"].quantile(0.95)]
top5_share = (outliers["total_revenue_birr"].sum() /
              df["total_revenue_birr"].sum()) * 100

# Results
print(f"✅ Rainfall impact (%): {rainfall_percent:.2f}")
print(f"✅ Fertilizer impact (%): {fert_percent:.2f}")
print(f"✅ Top 10% revenue share (%): {top10_share:.2f}")
print(f"✅ Top 5% revenue share (%): {top5_share:.2f}")

#🧠 MORE ADVANCED PART ⭐

#🧠 1. MULTI-DIMENSIONAL ANALYSIS (3+ VARIABLES)
multi = df.groupby(
    ["rainfall_category", "soil_type", "fertilizer_use"],
    observed=False
)["revenue_per_hectare"].mean().reset_index()

multi.sort_values(by="revenue_per_hectare", ascending=False).head(10)

print("\n✅=== Multi-Dimensional Top 10 ===")
print(multi.sort_values(by="revenue_per_hectare", ascending=False).head(10))

#🧠 2. ADVANCED CROSS-TAB (WITH MEAN PERFORMANCE)
cross = pd.pivot_table(
    df,
    values="revenue_per_hectare",
    index="rainfall_category",
    columns="soil_type",
    aggfunc="mean",
    observed=False
)

print("\n✅=== Cross-tab (Rainfall vs Soil) ===")
print(cross)

#🧠 3. SEGMENTATION (HIGH vs LOW PERFORMANCE FARMS)
seg = df["performance_segment"] = pd.qcut(
    df["revenue_per_hectare"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df.groupby(
    "performance_segment",
    observed=False
)[["rainfall_mm", "fertilizer_use", "land_hectares"]].mean()

print("\n✅=== Segmentation Analysis ===")
print(seg)

#🧠 4. COHORT ANALYSIS (REGION-BASED PERFORMANCE)
cohort = df.groupby("region")["revenue_per_hectare"].mean().sort_values(ascending=False)

print("\n✅=== Regional Performance (Cohort) ===")
print(cohort)

#🧠 5. CORRELATION (STRONG RELATIONSHIPS)
corr = df[[
    "rainfall_mm",
    "fertilizer_use",
    "land_hectares",
    "yield_per_hectare",
    "revenue_per_hectare"
]].corr()

print("\n✅=== Correlation Matrix ===")
print(corr)

#🧠 6. OUTLIER CHARACTERISTICS (NOT JUST FINDING THEM)
outliers = df[df["revenue_per_hectare"] > df["revenue_per_hectare"].quantile(0.95)]

outliers.describe()
print("\n✅=== Outlier Summary (Top 5%) ===")
print(outliers.describe())

#🧠 7. RATIO ANALYSIS (EFFICIENCY METRICS)
df["input_efficiency"] = df["yield_per_hectare"] / (df["fertilizer_use"] + 1)

eff = df.groupby(
    "rainfall_category",
    observed=False
)["input_efficiency"].mean()

print("\n✅=== Input Efficiency by Rainfall ===")
print(eff)

#🧠 8. STATISTICAL VALIDATION (ADVANCED ⭐)
from scipy.stats import ttest_ind

low_group = df[df["rainfall_category"] == "Low"]["revenue_per_hectare"]
high_group = df[df["rainfall_category"] == "High"]["revenue_per_hectare"]

t_stat, p_value = ttest_ind(low_group, high_group)

print("\n✅=== Statistical Test ===")
print(f"✅ T-statistic: {t_stat:.4f}")
print(f"✅ P-value: {p_value:.6e}")



✅ Rainfall impact (%): 57.68
✅ Fertilizer impact (%): 17.84
✅ Top 10% revenue share (%): 18.27
✅ Top 5% revenue share (%): 8.84

✅=== Multi-Dimensional Top 10 ===
   rainfall_category soil_type  fertilizer_use  revenue_per_hectare
14              High      Loam               0         82255.364588
15              High      Loam               1         81114.457108
13              High      Clay               1         76300.665801
12              High      Clay               0         72973.399527
17              High     Sandy               1         65147.942620
9             Medium      Loam               1         64047.651955
11            Medium     Sandy               1         60848.616105
8             Medium      Loam               0         60076.480731
7             Medium      Clay               1         59507.232282
16              High     Sandy               0         57547.754953

✅=== Cross-tab (Rainfall vs Soil) ===
soil_type                  Clay          Loam     

🧠 Key Insights ⭐
🧠 1. Climate Impact (Strong + Validated): 

Revenue per hectare increases by 57.68% from low to high rainfall regions, and this difference is statistically significant (p = 2.52×10⁻³⁰), confirming that rainfall is a highly reliable driver of productivity. However, since the top 10% of farms generate only 18.27% of total revenue, rainfall alone does not create top performers, indicating that its effect depends on interaction with other factors such as soil and efficiency.

🧠 2. Environmental Dominance over Inputs:

Fertilizer increases productivity by 17.84%, which is significantly lower than the 57.68% rainfall impact, meaning rainfall has over 3.2× greater influence. This is further supported by correlation results, where rainfall shows a stronger relationship with revenue (r = 0.45) than fertilizer (r = 0.097), confirming that environmental conditions dominate input-based interventions.

🧠 3. Multi-Dimensional Optimization (Highest Performance Combination):

The highest revenue per hectare (82,255 birr) is achieved under high rainfall + loam soil + no fertilizer, while similarly high performance (81,114 birr) occurs with fertilizer, indicating that soil quality and rainfall can substitute or amplify input usage, and optimal performance depends on specific combinations rather than single variables.

🧠 4. Soil–Rainfall Interaction (Cross-tab Insight):

Under high rainfall, loam soil achieves the highest average revenue (81,636 birr) compared to clay (74,745 birr) and sandy (62,077 birr), representing a 31% advantage over sandy soil, demonstrating that soil type significantly moderates the impact of rainfall on productivity.

🧠 5. Structural Distribution of Productivity:

The top 10% of farms generate 18.27% of total revenue, while the top 5% contribute 8.84%, indicating that agricultural output is relatively evenly distributed. This suggests that productivity differences are driven by broad systemic factors rather than extreme inequality or a small elite group.

🧠 6. Outlier Characteristics (Quantified Efficiency Drivers):

The top 5% farms (40 farms) operate under high rainfall (mean = 1421 mm), achieve high yield (mean = 4.82 tons/hectare), and show moderate fertilizer use (65% adoption rate), indicating that high productivity is associated with favorable environmental conditions combined with efficient, but not excessive, input usage.

🧠 7. Efficiency Scaling with Rainfall:

Input efficiency increases from 2.02 (low rainfall) to 3.22 (high rainfall), representing a ~60% improvement, showing that inputs become significantly more productive under favorable environmental conditions, reinforcing that rainfall enhances not just output, but also input effectiveness.

🧠 8. Regional Inequality (Cohort Analysis):

Regional performance varies significantly, with Gambela (64,526 birr) outperforming Amhara (55,717 birr) by approximately 16%, indicating that geographic and structural factors such as climate and infrastructure create measurable regional productivity gaps.

🧠 9. Production Mechanism (Correlation Insight):

Rainfall has a strong correlation with yield (r = 0.77) and a moderate correlation with revenue (r = 0.45), while yield itself correlates strongly with revenue (r = 0.51), indicating that rainfall primarily influences productivity through increasing yield, which then drives revenue, revealing an indirect causal pathway.

🧠 10. Land Size Irrelevance (Non-Obvious Insight):

Land size shows almost no correlation with revenue (r = -0.038), indicating that larger farms do not necessarily generate higher productivity per hectare, suggesting that efficiency and environmental factors are more important than scale.

Note:
All insights are derived from computed percentage-based and statistical metrics, including correlation and hypothesis testing, ensuring that findings are supported by data and interpreted with awareness of multi-variable interactions and causality limitations.
